# New Notebook file

In [ ]:
import gc
import sys
import random
import torch
import heapq
import numpy as np
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from typing import List, Dict, Tuple
from collections import Counter
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import IterableDataset, DataLoader
from pathlib import Path
torch.manual_seed(42)
random.seed(42)
MIN_COUNT = 5
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
raw_tokens = open("data/text8").read().split()
raw_tokens[:3]

In [ ]:
print("Total Tokens ", len(raw_tokens))
tokens_dict = Counter(raw_tokens)
tokens = [tkn for tkn, count in tokens_dict.items() if count >= MIN_COUNT]
word_to_idx = {tkn: i for i, tkn in enumerate(tokens)}
idx_to_word = {i: tkn for i, tkn in enumerate(tokens)}
counts = [0]*len(word_to_idx)
print("Filtered Tokens ", len(tokens))
for tkn, i in word_to_idx.items():
    counts[i] = tokens_dict.get(tkn)
print("Total Words ", sum(counts))
del tokens_dict
gc.collect()
assert len(counts) == len(word_to_idx)

In [ ]:
71290/16718844

In [ ]:

V = len(word_to_idx)
stream = np.fromiter(
    (word_to_idx[t] for t in raw_tokens if t in word_to_idx),
    dtype=np.int32
)
del raw_tokens
gc.collect()
#len(stream)

In [ ]:
# Huffman Tree algorithm
class Node:

    def __init__(self, freq, word=None, left=None, right=None, node_id=None):
        self.freq = freq
        self.word = word
        self.left = left
        self.right = right
        self.node_id = node_id

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(counts, vocab_size):
    heap = [Node(count, word=idx) for idx, count in enumerate(counts)]
    heapq.heapify(heap)
    node_id = 0
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        parent = Node(left.freq + right.freq, left=left, right=right, node_id=node_id)
        node_id += 1
        heapq.heappush(heap, parent)
    assert heap[0].node_id == vocab_size - 2
    return heap[0]

def dfs(root: Node, current_path: List[int], path: List[List[int]], code: List[List[bool]], current_path_codes: List[bool]):
    if root is None:
        return

    # Leaf node check (Huffman leaf contains the character/word)
    if root.left is None and root.right is None:
        # Create a shallow copy of current_path so future changes don't overwrite it
        path[root.word] = list(current_path)
        code[root.word] = list(current_path_codes)
        return

    # 1. Add current node to path
    current_path.append(root.node_id)
    current_path_codes.append(0)

    # 2. Recursively search left and right branches
    dfs(root.left, current_path, path, code, current_path_codes)
    current_path_codes.pop()
    current_path_codes.append(1)
    dfs(root.right, current_path, path, code, current_path_codes)
        
    # 3. Backtrack: remove node and unvisit for other paths
    current_path.pop()
    current_path_codes.pop()

root = build_huffman_tree(counts, V)
paths = [None]*V
codes = [None]*V
dfs(root, [], paths, codes, [])
prints = 0
for word_id, route in enumerate(paths):
    pth = ""
    for idx, node_id in enumerate(route):
        pth = f"{pth} -> {codes[word_id][idx]} -> {node_id}"
    print(f"{word_id}{pth}")
    prints += 1
    if prints >= 5:
        break

In [ ]:
min_path_len = min(len(pth) for pth in paths)
max_path_len = max(len(pth) for pth in paths) 
print(min_path_len, max_path_len)

In [ ]:
masks = [None]*len(paths)
for idx, path in enumerate(paths):
    mask = [1]*len(path)
    path = path + [0]*(max_path_len - len(path))
    code = codes[idx]
    code = code + [0]*(max_path_len-len(code))
    mask = mask + [0]*(max_path_len - len(mask))
    masks[idx] = mask
    paths[idx] = path
    codes[idx] = code
assert all(len(path) == max_path_len for path in paths), "Not all path lenghts are 22"
assert all(len(code) == max_path_len for code in codes), "Not all code lengths are 22"
paths = torch.tensor(paths, dtype=torch.int32)
codes = torch.tensor(codes, dtype=torch.float32)
masks = torch.tensor(masks, dtype=torch.float32)

In [ ]:
# Constants
C = 5
B = 1
D = 100
num_epochs = 1 
learning_rate = 1

In [ ]:
class SkipGramDataset(IterableDataset):

    def __init__(self, stream, max_window_size):
        self.stream = stream
        self.max_window_size = max_window_size

    def __iter__(self):
        n = len(self.stream)
        for center_pos in range(n):
            radius = random.randint(1, self.max_window_size) 
            left = max(0, center_pos - radius)
            right = min(n, center_pos + radius+1)
            for context_pos in range(left, right):
                if center_pos == context_pos:
                    continue
                yield (
                    self.stream[center_pos],
                    self.stream[context_pos]
                )

class CBOWDataset(IterableDataset):

    def __init__(self, stream, max_window_size):
        self.stream = stream
        self.max_window_size = max_window_size
        # widest possible context: max_window_size words on each side
        self.max_context = 2 * max_window_size

    def __iter__(self):
        n = len(self.stream)
        for center_pos in range(n):
            radius = random.randint(1, self.max_window_size)
            left = max(0, center_pos - radius)
            right = min(n, center_pos + radius + 1)
            # order does not matter for CBOW (the mean destroys it), so the two
            # sides are concatenated and the whole row is padded at the end.
            # Padding at the end only is what keeps context_mask aligned.
            context = np.concatenate([
                self.stream[left:center_pos],
                self.stream[center_pos + 1:right],
            ])
            n_real = len(context)
            assert 0 < n_real <= self.max_context, f"n_real={n_real}"
            pad = self.max_context - n_real
            yield (
                np.concatenate([context, np.zeros(pad, dtype=self.stream.dtype)]),
                self.stream[center_pos],
                np.concatenate([np.ones(n_real, dtype=np.float32),
                                np.zeros(pad, dtype=np.float32)]),
            )


# MAX_EXP: word2vec.c skips any node whose logit falls outside [-6, +6].
# torch.clamp reproduces it exactly - the value is bounded AND the gradient is
# zero outside the range, so an out-of-range node updates neither word_emb nor
# node_emb. Without it the bilinear HS objective runs away: |h| grows with
# |node_vec| and |node_vec| grows with |h|, so past a threshold the growth is
# exponential and no learning rate is safe. Verified no-op at init (all logits
# are 0 there), so the 7.3808 init checkpoint is unaffected.
MAX_EXP = 6.0


class SkipGram(nn.Module):
    def __init__(self, vocab_size, emb_dim, paths, codes, masks, max_exp=MAX_EXP):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.max_exp = max_exp
        self.word_emb = nn.Embedding(vocab_size, emb_dim)
        self.node_emb = nn.Embedding(vocab_size-1, emb_dim)
        bound = 0.5/emb_dim
        nn.init.uniform_(self.word_emb.weight, -bound, bound)
        nn.init.zeros_(self.node_emb.weight)
        self.register_buffer(
            "paths",
            paths
        )
        self.register_buffer(
            "codes",
            codes
        )
        self.register_buffer(
            "masks",
            masks
        )
       

    def forward(self, centers, targets):
        targets = targets.long()
        centers_emb = self.word_emb(centers)
        assert centers_emb.shape == (centers.shape[0], self.emb_dim)
        context_emb = self.node_emb(self.paths[targets])
        logits = (context_emb * centers_emb.unsqueeze(1)).sum(dim=-1)
        code = self.codes[targets]
        mask = self.masks[targets]
        assert logits.shape == code.shape
        if self.max_exp:
            logits = logits.clamp(-self.max_exp, self.max_exp)
        logits = logits*mask
        per_node = F.binary_cross_entropy_with_logits(
            logits, code, reduction="none"
        )
        return (per_node*mask).sum(dim=1).mean()


class CBOW(nn.Module):

    def __init__(self, vocab_size, emb_dim, paths, codes, masks, max_window_size,
                 max_exp=MAX_EXP):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.max_exp = max_exp
        self.word_emb = nn.Embedding(vocab_size, emb_dim)
        self.node_emb = nn.Embedding(vocab_size - 1, emb_dim)
        self.input_len = 2*max_window_size
        bound = 0.5/emb_dim
        nn.init.uniform_(self.word_emb.weight, -bound, bound)
        nn.init.zeros_(self.node_emb.weight)
        self.register_buffer(
            "paths",
            paths
        )
        self.register_buffer(
            "codes",
            codes
        )
        self.register_buffer(
            "masks",
            masks
        )

    def forward(self, contexts, targets, context_mask):
        # NOTE: here `targets` are the CENTER words. In SkipGram they were the
        # context words. Same head, opposite direction.
        # contexts = (B, input_len), targets = (B,), context_mask = (B, input_len)
        contexts, targets = contexts.long(), targets.long()
        batch = contexts.shape[0]

        context_emb = self.word_emb(contexts)
        assert context_emb.shape == (batch, self.input_len, self.emb_dim)

        # masked mean: numerator drops the padded slots, denominator counts
        # only the real ones. keepdim=True so (batch, 1) broadcasts over D.
        h = (
            (context_emb * context_mask.unsqueeze(-1)).sum(dim=1)
            / context_mask.sum(dim=1, keepdim=True)
        )
        assert h.shape == (batch, self.emb_dim)

        # everything below is identical to SkipGram.forward
        node_vecs = self.node_emb(self.paths[targets])
        logits = (node_vecs * h.unsqueeze(1)).sum(dim=-1)
        code = self.codes[targets]
        mask = self.masks[targets]
        assert logits.shape == code.shape == mask.shape

        if self.max_exp:
            logits = logits.clamp(-self.max_exp, self.max_exp)
        logits = logits*mask
        per_node = F.binary_cross_entropy_with_logits(
            logits, code, reduction="none"
        )
        return (per_node*mask).sum(dim=1).mean()

In [ ]:
LN2 = float(np.log(2))


def build(kind, batch_size):
    """Fresh model + fresh loader. Seeds are set here so every call starts from
    identical weights AND an identical pair stream."""
    random.seed(42)
    torch.manual_seed(42)
    if kind == "skipgram":
        model = SkipGram(V, D, paths, codes, masks)
        loader = DataLoader(SkipGramDataset(stream, max_window_size=C), batch_size=batch_size)
    elif kind == "cbow":
        model = CBOW(V, D, paths, codes, masks, C)
        loader = DataLoader(CBOWDataset(stream, max_window_size=C), batch_size=batch_size)
    else:
        raise ValueError(f"unknown kind {kind!r}")
    return model.to(device), loader


def split_batch(kind, batch):
    """-> (forward args on device, hs_target on cpu).

    hs_target is whichever word the hierarchical-softmax head has to predict:
    the CONTEXT word for skip-gram, the CENTER word for CBOW. predicted_loss
    must be computed from that word's path depth, nothing else.
    """
    if kind == "skipgram":
        centers, contexts = batch
        centers, contexts = centers.long(), contexts.long()
        return (centers.to(device), contexts.to(device)), contexts
    contexts, centers, cmask = batch
    contexts, centers = contexts.long(), centers.long()
    return (contexts.to(device), centers.to(device), cmask.to(device)), centers


def run(lr, kind="skipgram", batch_size=16, max_steps=2000, decay=False, log_every=500):
    model, loader = build(kind, batch_size)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    # off for sweeps so lr is the only variable; on for the long run
    scheduler = (
        LambdaLR(optimizer, lambda step: max(1e-4, 1 - step / max_steps))
        if decay else None
    )

    observed_losses, predicted_losses, diff = [], [], []
    total_loss, num_batches = 0.0, 0
    for batch_idx, batch in enumerate(loader):
        if batch_idx >= max_steps:
            break
        args, hs_target = split_batch(kind, batch)
        # zero-learning prediction: avg path depth of this batch's HS targets * ln2
        predicted_loss = masks[hs_target].sum(dim=1).mean().item() * LN2
        loss = model(*args)
        observed_losses.append(loss.item())
        predicted_losses.append(predicted_loss)
        diff.append(loss.item() - predicted_loss)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        num_batches += 1
        if batch_idx % log_every == 0:
            print(f"step {batch_idx}  obs {loss.item():.4f}  pred {predicted_loss:.4f}  diff {diff[-1]:+.4f}")
    print(f"{kind} lr={lr} B={batch_size}: mean loss {total_loss/num_batches:.4f} over {num_batches} steps")
    return model, predicted_losses, observed_losses, diff


def run_sweep(lrs: List, kind="skipgram", batch_size=16, max_steps=1000):
    results = {}
    for lr in lrs:
        print(f"--- {kind}  B={batch_size}  lr={lr} ---")
        results[lr] = run(lr, kind=kind, batch_size=batch_size, max_steps=max_steps)
    print(f"\n{'lr':>7}  {'mean diff (last 200)':>22}  {'max |node_emb[0]|':>18}")
    for lr in lrs:
        model, _, _, diff = results[lr]
        tail = diff[-200:]
        # node 0 trains only from padding slots (mask=0 -> zero grad), so a
        # nonzero value here means NaN/inf leaked into every parameter
        max_emb = model.node_emb.weight[0].abs().max().item()
        print(f"{lr:>7}  {sum(tail)/len(tail):>+22.6f}  {max_emb:>18.4g}")
    return results

In [ ]:
# lr sweep at the batch size the long run will use.
# B=64's edge was between 5 and 8; smaller B averages fewer gradients, so less
# cancellation, so the workable lr should come DOWN by roughly sqrt(64/16) = 2.
sg_results = run_sweep([1, 2, 4, 8], kind="skipgram", batch_size=16, max_steps=1000)

In [ ]:
# CBOW needs its own sweep - skip-gram's lr will NOT transfer.
# h is a mean over ~5 context words, so each word_emb row receives its gradient
# divided by n_real on top of the batch division. Expect a higher workable lr.
cbow_results = run_sweep([1, 4, 8, 16], kind="cbow", batch_size=16, max_steps=1000)

In [ ]:
PROPER_NOUN_SYN = "gram6-nationality-adjective"
# categories whose words are named entities, regardless of the sem/syn label
NAMED_ENTITY = {
    "capital-common-countries",
    "capital-world",
    "city-in-state",
    "currency",
    PROPER_NOUN_SYN,
}

In [ ]:
rng = np.random.default_rng(seed=42)
OFFS = np.array([o for o in range(-5, 6) if o != 0])

def get_random_word_positions(rng, stream, word, sample_size, word_to_idx):
    positions = np.where(stream == word_to_idx[word])
    return rng.choice(positions[0], size=sample_size, replace=False)

def get_neighbour_word_count(stream, positions, idx_to_word):
    idx = positions[:, None] + OFFS[None, :]
    idx = idx[(idx >= 0) & (idx < len(stream))]
    freq_count = np.bincount(stream[idx])
    top_20_indices = np.argsort(freq_count)[-20:][::-1]
    top_20_values = freq_count[top_20_indices]
    return {idx_to_word[idx]: int(freq_count[idx]) for idx in top_20_indices}

def get_top_20_words_counts_for_word(word, sample_size, rng, stream, word_to_idx, idx_to_word):
    positions = get_random_word_positions(rng, stream, word, sample_size, word_to_idx)
    return get_neighbour_word_count(stream, positions, idx_to_word)



def get_word_positions(stream, word_to_idx):
    # rng = np.random.default_rng()
    sorter = np.argsort(stream)
    REPO = Path().resolve()
    data_path = REPO / "data" / "questions-words.txt"
    data = data_path.read_text().split("\n")
    current_cat = None
    current_set = {}
    named_entity_word_positions = {}
    common_words_positions = {}
    for line in data:
        if not line.strip():
            continue
        if line.startswith(":"):
            current_cat = line[1:].strip()
            continue
        words = line.lower().strip().split()
        current_set = named_entity_word_positions if current_cat in NAMED_ENTITY else common_words_positions
        if len(words) != 4:
            continue
        for word in words:
            if word in current_set:
                continue
            if word not in word_to_idx:
                continue
            idx = np.int32(word_to_idx[word])   # ya stream.dtype.type(...)
            left  = np.searchsorted(stream, idx, sorter=sorter, side='left')
            right = np.searchsorted(stream, idx, sorter=sorter, side='right')
            current_set[word] = sorter[left:right]
    return common_words_positions, named_entity_word_positions

def entropy(word_counts):
    counts = word_counts[word_counts > 0]
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities))


def calculate_neighbour_entropy(rng, stream, positions, N):
    p = rng.choice(positions, size=N, replace=False)
    idx = p[:, None] + OFFS[None, :]
    idx = idx[(idx >= 0) & (idx < len(stream))]   # stream ke kinaare sambhal lo
    return entropy(np.bincount(stream[idx]))


common_words_positions, named_entity_word_positions = get_word_positions(stream, word_to_idx)
N = [50, 100, 200]
common_words_entropy = []
named_entity_words_entropy = []
entropies = {n: {"common": [], "named_entity": []} for n in N}
for n, d in entropies.items():
    for _, positions in common_words_positions.items():
        if positions.shape[0] < n:
            continue
        entropies[n]['common'].append(calculate_neighbour_entropy(rng, stream, positions, n))
    for _, positions in named_entity_word_positions.items():
        if len(positions) < n:
            continue
        entropies[n]['named_entity'].append(calculate_neighbour_entropy(rng, stream, positions, n))




In [ ]:
a = np.array([-2, -1, 0, 1, 2])
p = rng.choice([1,2,3,4,5,6,7], size=6, replace=False)
idx = p[:, None] + a[None, :]
print(a[None, :], p[:, None])
idx

In [ ]:
freq_count = np.bincount(stream)
top_100_indices = np.argsort(freq_count)[-100:][::-1]
top_100_values = freq_count[top_100_indices]
stream_trimmed = stream[~np.isin(stream, top_100_indices)]
athens = get_top_20_words_counts_for_word('athens', 200, rng, stream_trimmed, word_to_idx, idx_to_word)
walking = get_top_20_words_counts_for_word('walking', 200, rng, stream_trimmed, word_to_idx, idx_to_word)
# corpus = get_neighbour_word_count(stream_trimmed, rng.choice(stream_trimmed, size=200, replace=False), idx_to_word)
print("Athens->", athens)
print("walking->", walking)
# print("Corpus->", corpus)

In [ ]:


#lking %%